### 1. Calculo de metricas
Para este notebook se requiere utilizar hasta Python 3.10, para que la libreria AligScore se deben contar con versiones especificas que pueden hacer funcionar mal los procesos de fine tuning por lo que se dejan por separado, la idea es tomar todos los archivos csv que cuentan con el texto cientifico, texto resumen original y el resumen generado por medio del LLM en este notebook y realizar el calculo de las metricas: Legibilidad, Relevancia y Factualidad.

Instalacion libreria AlignScore:
https://github.com/yuh-zha/AlignScore

In [ ]:
!pip install --quiet  -r req-fine-models-metrics.txt

In [ ]:
import pandas as pd
import numpy as np
import textstat
from typing import List, Dict, Any, Optional, Tuple
from bert_score import score as bert_score
import torch
from pathlib import Path
DATA_ALIGN = Path("./models/alignscore")
DATA_ALIGN.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
### Modelo requerido base, puede utilizarse large tambien, podria descargarse de HuggingFace, en una proxima revision lo ajusto.

!(cd models/alignscore; curl -O -OL https://huggingface.co/yzha/AlignScore/resolve/main/AlignScore-base.ckpt)
#!(cd models/alignscore; curl -O -OL https://huggingface.co/yzha/AlignScore/resolve/main/AlignScore-large.ckpt)


In [ ]:
def calcular_factualidad_alignscore(preds, refs,evaluation_mode, batch_size, device,flag_threshold: float = 0.5):
#evaluation_mode,    # 'nli_sp' (por defecto AlignScore), 'nli', 'bin_sp', 'bin'
    assert len(preds) == len(refs), "preds y refs deben tener la misma longitud"

    # Import tardío para que esta función siga importando aunque no esté instalada la lib.
    from alignscore import AlignScore  
    # Inicializar scorer
    backbone = 'roberta-base'
    scorer = AlignScore(
        model="roberta-base",
        batch_size=batch_size,
        device=device,
        ckpt_path='models/alignscore/AlignScore-large.ckpt',
        evaluation_mode=evaluation_mode
    )

    scores = scorer.score(contexts=refs, claims=preds) 
    scores = [float(s) for s in scores]

    flags_low = [bool(s < flag_threshold) for s in scores]
    per_example = pd.DataFrame({"alignscore": scores,"flag_low": flags_low}) 

    summary = {
        "mean_alignscore": float(np.mean(scores)) if scores else float("nan"),
        "std_alignscore":  float(np.std(scores)) if scores else float("nan"),
        "min_alignscore":  float(np.min(scores)) if scores else float("nan"),
        "max_alignscore":  float(np.max(scores)) if scores else float("nan"),
        "n_examples":      int(len(scores)),
        "backbone":        backbone,
        "evaluation_mode": evaluation_mode,
        "batch_size":      int(batch_size),
        "device":          device,
        "ckpt_path":       'models/alignscore/AlignScore-base.ckpt',
        "flag_threshold":  float(flag_threshold)
    }

    return summary, per_example



In [ ]:
def calcular_bertscore_relevancia(preds,refs,idf,rescale_with_baseline,batch_size,device):

    modelo = "roberta-base"
    assert len(preds) == len(refs), "preds y refs deben tener la misma longitud"


    P, R, F1 = bert_score(
        cands=preds.tolist(),
        refs=refs.tolist(),
        lang='en',
        model_type=modelo,
        idf=idf,
        rescale_with_baseline=rescale_with_baseline,
        batch_size=batch_size,
        device=device
    )

    p_list = [float(p) for p in P]
    r_list = [float(r) for r in R]
    f1_list = [float(f) for f in F1]

    summary = {
        "mean_precision": float(np.mean(p_list)) if p_list else float("nan"),
        "mean_recall":    float(np.mean(r_list)) if r_list else float("nan"),
        "mean_f1":        float(np.mean(f1_list)) if f1_list else float("nan"),
        "backbone_for_bertscore": modelo,
        "idf": bool(idf),
        "rescale_with_baseline": bool(rescale_with_baseline),
        "batch_size": int(batch_size),
        "device": device if device is not None else "auto"
    }

    per_example = {
        "bertscore_precision": p_list,
        "bertscore_recall": r_list,
        "bertscore_f1": f1_list
    }

    per_example = pd.DataFrame(per_example)

    return summary, per_example


In [ ]:
def calcular_legibilidad_textstat(preds):
    lang = 'en'
    textstat.set_lang(lang)
    rows = []
    for t in preds:
        t = t or ""

        row = {
            "flesch_reading_ease":  float(textstat.flesch_reading_ease(t)),
            "flesch_kincaid_grade": float(textstat.flesch_kincaid_grade(t)),
        }
        row.update({
            "gunning_fog":              float(textstat.gunning_fog(t)),
            "smog_index":               float(textstat.smog_index(t)) if textstat.sentence_count(t) >= 3 else float("nan"),
            "dale_chall":               float(textstat.dale_chall_readability_score(t)),
            "automated_readability":    float(textstat.automated_readability_index(t)),
            "coleman_liau":             float(textstat.coleman_liau_index(t)),
            "text_standard":            textstat.text_standard(t, float_output=True),
            "num_sentences":            int(textstat.sentence_count(t)),
            "num_words":                int(textstat.lexicon_count(t, removepunct=True)),
            "syllables":                int(textstat.syllable_count(t)),
            "reading_time_sec":         float(textstat.reading_time(t)),
        })

        rows.append(row)

    def _try_mean(key: str):
        vals = [r[key] for r in rows if key in r and isinstance(r[key], (int, float)) and not np.isnan(r[key])]
        return float(np.mean(vals)) if vals else float("nan")

    keys = sorted({k for r in rows for k in r.keys()})
    summary = {"n_examples": len(preds), "lang": lang}
    for k in keys:
        summary[f"mean_{k}"] = _try_mean(k)

    per_example = pd.DataFrame(rows) 
    return summary, per_example


In [ ]:
def calcular_metricas(df):
    print('**** Inicio relevancia')
    summary_relevancia, per_example_relevancia = calcular_bertscore_relevancia(
        df['gen_summary'],df['article'],
        idf=True,#Set pequeno a false, sino dejar en TRUE
        rescale_with_baseline=False,
        batch_size=1,
        device=device
    )
    print('**** Fin relevancia')
    print('**** Inicio legibilidad')
    summary_legibilidad, per_example_legibilidad = calcular_legibilidad_textstat(df['gen_summary'])
    print('**** Fin legibilidad')
    print('**** Inicio factualidad')
    summary_factualidad, per_example_factualidad = calcular_factualidad_alignscore(df['gen_summary'], df['article'],'nli_sp', 16, device)
    print('**** Fin factualidad')
    return summary_relevancia, summary_legibilidad,summary_factualidad


### Calculo de ejemplo de las 3 metricas requeridas para llama3

In [ ]:
data = pd.read_csv('models/results/summaries_llama3.2-1b.csv')
sum_relevancia, sum_legibilidad,sum_factualidad = calcular_metricas(data)

In [ ]:
print(sum_relevancia)
print(sum_legibilidad)
print(sum_factualidad)

In [ ]:
metricas = pd.concat({
    "relevancia": pd.json_normalize(sum_relevancia, sep="__"),
    "legibilidad": pd.json_normalize(sum_legibilidad, sep="__"),
    "factualidad": pd.json_normalize(sum_factualidad, sep="__"),
}, axis=1)

metricas.to_csv("models/results/metrics_llama3.2-1b.csv", index=False)